In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


In [49]:
df = pd.read_pickle("filtered_data.pkl")

In [50]:
df.index.unique()

DatetimeIndex(['2018-06-19 00:00:00+00:00', '2018-06-20 00:00:00+00:00',
               '2018-06-21 00:00:00+00:00', '2018-06-22 00:00:00+00:00',
               '2018-06-25 00:00:00+00:00', '2018-06-26 00:00:00+00:00',
               '2018-06-27 00:00:00+00:00', '2018-06-28 00:00:00+00:00',
               '2018-06-29 00:00:00+00:00', '2018-07-02 00:00:00+00:00',
               ...
               '2026-07-27 00:00:00+00:00', '2026-07-28 00:00:00+00:00',
               '2026-07-29 00:00:00+00:00', '2026-07-30 00:00:00+00:00',
               '2026-07-31 00:00:00+00:00', '2026-08-03 00:00:00+00:00',
               '2026-08-04 00:00:00+00:00', '2026-08-05 00:00:00+00:00',
               '2026-08-06 00:00:00+00:00', '2026-08-07 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='ts_event', length=2045, freq=None)

In [51]:
df.reset_index(inplace=True)


In [52]:
indice_dict = {'XLE':"Energy", 
               'XLB':"Materials", 
               'XLI':"Industrials", 
               'XLY':"Consumer Discretionary", 
               'XLP':"Consumer Staples", 
               'XLV':"Healthcare", 
               'XLF':"Financials", 
               'XLK':"Technology", 
               'XLC':"Communication Services", 
               'XLU':"Utilities", 
               'XLRE':"Real Estate"
               }

In [53]:
px   = df.pivot(index='ts_event', columns='symbol', values='close').rename(columns=indice_dict)
rets = px.pct_change().dropna()

In [54]:
mkt  = rets.mean(axis=1)
rel  = rets.sub(mkt, axis=0).iloc[:, :-1]
X    = sm.add_constant(pd.concat([mkt.rename('MKT'), rel], axis=1))


In [55]:
port = pd.read_pickle('8YearsData.pkl')
port = port[port['symbol'] == 'NVDA']['close'].pct_change()


In [56]:
data = pd.concat([port.rename('port'), X], axis=1).dropna()
Xf   = data.drop(columns='port')
fit  = sm.OLS(data['port'], Xf).fit()

In [57]:
b        = fit.params.drop('const')
F        = Xf.drop(columns='const').cov() * 252
sys_var  = float(b @ F @ b)
idio_var = fit.resid.var() * 252

In [58]:
print(f"Obs: {fit.nobs:.0f}   {data.index.min().date()} to {data.index.max().date()}")
print(f"R²: {fit.rsquared:.3f}   Adj R²: {fit.rsquared_adj:.3f}")
print(f"Dropped sector (base case): {[c for c in rets.columns if c not in Xf.columns][0]}\n")

print(f"Alpha (ann):     {fit.params['const']*252:>8.2%}   t = {fit.tvalues['const']:.2f}")
print(f"Predicted vol:   {np.sqrt(sys_var+idio_var):>8.2%}")
print(f"Realized vol:    {data['port'].std()*np.sqrt(252):>8.2%}")
print(f"Factor vol:      {np.sqrt(sys_var):>8.2%}")
print(f"Idio vol:        {np.sqrt(idio_var):>8.2%}")
print(f"Factor % of var: {sys_var/(sys_var+idio_var):>8.1%}\n")

out = pd.DataFrame({
    'beta':      b,
    't_stat':    fit.tvalues.drop('const'),
    'ret_contr': b * Xf.drop(columns='const').mean() * 252,
    'risk_%':    b * (F @ b) / sys_var,
}).sort_values('risk_%', ascending=False)
print(out.to_string(float_format=lambda v: f"{v:9.3f}"))

Obs: 2044   2018-06-20 to 2026-08-07
R²: 0.350   Adj R²: 0.347
Dropped sector (base case): Consumer Discretionary

Alpha (ann):        5.47%   t = 0.29
Predicted vol:     66.61%
Realized vol:      66.61%
Factor vol:        39.41%
Idio vol:          53.69%
Factor % of var:    35.0%

                            beta    t_stat  ret_contr    risk_%
Technology                 1.516    10.229      0.136     0.566
MKT                        1.129    14.112      0.099     0.353
Utilities                 -0.461    -3.715      0.028     0.088
Communication Services     0.474     3.481      0.017     0.059
Industrials                0.386     2.671      0.019     0.002
Materials                 -0.322    -2.090      0.017     0.001
Financials                 0.078     0.575      0.002    -0.001
Energy                     0.016     0.159     -0.001    -0.001
Healthcare                 0.184     1.354      0.002    -0.018
Real Estate                0.189     1.439     -0.004    -0.021
Consumer Stap

In [59]:
tot_var = sys_var + idio_var

print(f"Obs: {fit.nobs:.0f}   {data.index.min().date()} to {data.index.max().date()}")
print(f"R²: {fit.rsquared:.3f}   Adj R²: {fit.rsquared_adj:.3f}\n")

print(f"Alpha (ann):     {fit.params['const']*252:>8.2%}   t = {fit.tvalues['const']:.2f}\n")

print(f"{'':<16}{'variance':>10}{'vol':>10}{'% of var':>10}")
print(f"{'Factor':<16}{sys_var:>10.4f}{np.sqrt(sys_var):>9.2%}{sys_var/tot_var:>10.1%}")
print(f"{'Idiosyncratic':<16}{idio_var:>10.4f}{np.sqrt(idio_var):>9.2%}{idio_var/tot_var:>10.1%}")
print(f"{'':-<46}")
print(f"{'Total':<16}{tot_var:>10.4f}{np.sqrt(tot_var):>9.2%}{1.0:>10.1%}")
print(f"{'Realized':<16}{data['port'].var()*252:>10.4f}{data['port'].std()*np.sqrt(252):>9.2%}\n")

print(f"Check — factor share {sys_var/tot_var:.4f} vs R² {fit.rsquared:.4f}")

Obs: 2044   2018-06-20 to 2026-08-07
R²: 0.350   Adj R²: 0.347

Alpha (ann):        5.47%   t = 0.29

                  variance       vol  % of var
Factor              0.1553   39.41%     35.0%
Idiosyncratic       0.2883   53.69%     65.0%
----------------------------------------------
Total               0.4436   66.61%    100.0%
Realized            0.4436   66.61%

Check — factor share 0.3501 vs R² 0.3501
